# Environment Check
Different nodes on Colab may have different versions of torch which can cause issues with the implimantation of the rest of the code. The following addresses these inconsistencies and addresses the issue.

In [ ]:
import torch
import os

# Check if PyTorch is correctly installed with CUDA support
try:
    if torch.cuda.is_available():
        print(f"PyTorch version {torch.__version__} with CUDA {torch.version.cuda} is already installed and working.")
    else:
        # If CUDA is not available, force a reinstall.
        raise ImportError("CUDA not available")
except (ImportError, AttributeError):
    # This block runs if CUDA is not available OR if we get the AttributeError
    print("PyTorch CUDA issue detected. Forcing re-installation...")

    # Uninstall existing versions to prevent conflicts
    !pip uninstall -y torch torchvision torchaudio

    # Install a known stable version of PyTorch compatible with Colab's CUDA 12.1
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

    print("\nRe-installation complete.")
    print("!!! IMPORTANT: You may need to RESTART the runtime now for the new installation to take effect. !!!")

    # A small piece of code to automatically crash the kernel, forcing a restart.
    import os
    os.kill(os.getpid(), 9)

PyTorch version 2.6.0+cu124 with CUDA 12.4 is already installed and working.


# Mount Google Drive

In [ ]:
# Mount Google Drive if running on colab
import os
from pathlib import Path
try:
  from google.colab import drive
  drive.mount('/content/drive')

  # Setting Current Working Directory to Project Folder
  PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/")
  os.chdir(PROJECT_ROOT)
  print(f"Current Directory: {os.getcwd()}")
  ON_COLAB = True

except ImportError: # Will trip if running locally
  print(f"Not running on google colab or Drive failed to mount. Assume CWD is root directory")
  PROJECT_ROOT = Path.cwd()
  ON_COLAB = False

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Current Directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Directory: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project
Current Directory: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project


# Clone Github Repository

In [ ]:
REPO_PATH = PROJECT_ROOT / "DeepRL-MsPacman"

if not REPO_PATH.exists():
  print("Cloning repository...")
  !git clone https://github.com/davisalexanderc/DeepRL-MsPacman.git {REPO_PATH}
  #!git clone --branch debug/memory-leak-info-dict https://github.com/davisalexanderc/DeepRL-MsPacman.git {REPO_PATH}
else:
  print(f"Repository already exists at {REPO_PATH}")
  %cd {REPO_PATH}
  !git checkout main
  !git pull

%cd {REPO_PATH}

Repository already exists at /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DeepRL-MsPacman
/content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DeepRL-MsPacman
D	~$Eval_Run_Logs.xlsx
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.
/content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DeepRL-MsPacman


# Uncomment on first run
The following is needed to use earlier versions on gymnasium and numpy so that all the libraries are able to interact correctly

In [ ]:
#!pip uninstall -y gymnasium numpy
#!pip install gymnasium==0.29.1 numpy==1.26.4

# Install Dependencies

In [ ]:
print("Installing dependencies...")
!python -m pip install -r requirements.txt
print("Installation complete!")

from train import train_agent
from common.utils import setup_environment_and_agent, find_checkpoints
from pathlib import Path
import time
from typing import Optional
import sys
if str(REPO_PATH) not in sys.path:
  sys.path.append(str(REPO_PATH))

Installing dependencies...
Installation complete!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# Run Parameters and Hyperparameters

In [ ]:
config = {
    "agent": "dqn",
    "total_timesteps": 30_000_000, # Keep constant at 10 Million
    "learning_starts": 50_000,     # Keep constant at 50 Thousand
    "train_frequency": 4,         # Train the model every 4 steps
    "target_update_frequency": 20_000, ### Try [1_000, 10_000* and 20_000]
    "save_frequency": 500_000,    # Save a model checkpoint every 500,000 steps
    "log_frequency": 10_000,      # Log progress every 10,000 steps

    # Agent specific
    "replay_buffer_capacity": 500_000,  # Upper limit (250_000) due to Colab RAM limits
    "batch_size": 32,
    "learning_rate": 0.0001,    ### Try [0.00005, 0.0001*, 0.00025]
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,  # Changed from 0.1 in earlier training
    "epsilon_decay_duration": 2_500_000, ### Try [500_000, 1_000_000*, 2_500_000]

    # Reward Shaping
    "enable_reward_shaping": False,
    "enable_level_completion_bonus": False,
    "level_completion_bonus": 5000.0,
    "enable_death_penalty": False,
    "death_penalty": -250.0,
    "enable_time_penalty": False,
    "time_penalty_per_step": -1,

    # Paths (using our Google Drive path)
    "save_path": PROJECT_ROOT / "checkpoints",
    "log_path": PROJECT_ROOT / "logs"
}

# --- Create the directories in Google Drive if they don't exist ---
config["save_path"].mkdir(parents=True, exist_ok=True)
config["log_path"].mkdir(parents=True, exist_ok=True)
print(f"Artifact directories created/verified at: {PROJECT_ROOT}")

Artifact directories created/verified at: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project


# Adjust here to restart a run from the last checkpoint

In [ ]:
RESUME_TRAINING = True
JOB_NAME = 'DQN_Run_14'
RESUME_PATH = PROJECT_ROOT / JOB_NAME

if RESUME_TRAINING:
    # Logic to set paths for resuming
    resume_path_obj = Path(RESUME_PATH)
    config["log_path"] = resume_path_obj
    config["save_path"] = resume_path_obj
    RESUME_PATH = resume_path_obj
else:
    print("Starting new training run")
    RESUME_PATH = None

In [ ]:
print("--- Creating Environment and Agent ---")
wrapped_env, agent, device = setup_environment_and_agent(config)

# If resuming, load the checkpoint now
if RESUME_TRAINING:
    checkpoint_paths = find_checkpoints(RESUME_PATH)
    if checkpoint_paths:
        latest_checkpoint = checkpoint_paths[-1]
        # We need to manually get the start_timestep and update the agent
        start_timestep = agent.load(latest_checkpoint) + 1
        if hasattr(agent, 'timestep'):
            agent.timestep = start_timestep
        config['start_timestep'] = start_timestep # Store it in the config for the next cell
    else:
        print("Warning: Resuming but no checkpoints found.")
        config['start_timestep'] = 1
else:
    config['start_timestep'] = 1

print("\n--- Setup Complete. Agent and config are now in memory. ---")
print(f"Agent will train from step {config['start_timestep']} to {config['total_timesteps']}.")

--- Creating Environment and Agent ---
Using device: cuda
Environment created and wrapped with unified AtariWrapper.
DQN Agent instantiated.
Searching for checkpoints in: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14
Loading checkpoint from /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_model_step_20000000.pth...
Loaded new-style checkpoint. Resuming from timestep 20000000.
Loading replay buffer from /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_model_step_20000000.pkl...
Loaded replay buffer with 500000 experiences.
Loaded new-style checkpoint. Resuming from timestep 20000000.

--- Setup Complete. Agent and config are now in memory. ---
Agent will train from step 20000001 to 30000000.


# Launch Training

In [9]:
from train import train_agent

print(f"--- Preparing to train agent: {config['agent']}")

train_agent(config, resume_path=RESUME_PATH)

--- Preparing to train agent: dqn
TensorBoard log directory: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_1755033937
Using device: cuda
Using device: cuda
Environment created and wrapped with unified AtariWrapper.
DQN Agent instantiated.
Searching for checkpoints in: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14
Loading checkpoint from /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_model_step_20000000.pth...
Loaded new-style checkpoint. Resuming from timestep 20000000.
Loading replay buffer from /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_model_step_20000000.pkl...
Loaded replay buffer with 500000 experiences.
Loaded new-style checkpoint. Resuming from timestep 20000000.
--- Starting Training ---


True Score: 2250.0, Ep Length: 867: 100%|███████████████████████████████████████████████████████████▉| 9999980/10000000 [16:27:22<00:00, 251.78step/s]

Final save at timestep 30000000. Including replay buffer in checkpoint.
Saving replay buffer separately to: /content/drive/MyDrive/Colab_Notebooks/AI572-RL/MsPacman_RL_Project/DQN_Run_14/dqn_model_step_30000000.pkl


True Score: 2250.0, Ep Length: 867: 100%|███████████████████████████████████████████████████████████| 10000000/10000000 [16:28:44<00:00, 168.56step/s]


--- Training Complete ---
